In [1]:
from pyspark.sql import functions as F

# Read ECB data from Bronze
fx_df = spark.read.table(
    "NYC_Mobility_Analytics.NYC_Bronze_LH.dbo.ecb_usd_eur_raw"
)

print("Original records:", fx_df.count())

# Clean and standardize
clean_fx_df = (
    fx_df
    .select(
        F.to_date("Date").alias("date"),
        F.col("USD").cast("double").alias("usd_per_eur")
    )
    .filter(F.col("date").isNotNull())
    .filter(F.col("usd_per_eur").isNotNull())
    .filter(F.col("usd_per_eur") > 0)
    .dropDuplicates(["date"])
    .orderBy("date")
)

# Add date attributes
clean_fx_df = (
    clean_fx_df
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
)

# Save to Silver Lakehouse
(
    clean_fx_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecb_usd_eur_clean")
)

print("ECB Silver data saved successfully!")
print("Clean records:", clean_fx_df.count())

display(clean_fx_df.orderBy(F.desc("date")).limit(10))

StatementMeta(, 469db7f6-5107-4d04-a38f-3dbb6689a9da, 3, Finished, Available, Finished, False)

Original records: 7127
ECB Silver data saved successfully!
Clean records: 7127


SynapseWidget(Synapse.DataFrame, 8fe2bcc1-0529-4681-bee8-edd21a46d892)